In [1]:
from dotenv import load_dotenv
load_dotenv()

from lib.udaplay_vector_store import UdaPlayVectorStore
from lib.udaplay_tools import configure_udaplay_tools
from lib.udaplay_agent import UdaPlayAgent
from lib.udaplay_reporting import format_json_report

### 1. Vector store

In [2]:
vector_store = UdaPlayVectorStore(
    persist_dir="./chroma_db/udaplay_games",
    collection_name="udaplay_games",
    reset_collection=False,
)


vector_store.add_games_from_file("data/games.json")

configure_udaplay_tools(vector_store)
agent = UdaPlayAgent(vector_store=vector_store)

print("UdaPlay agent ready")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

UdaPlay agent ready


### 2. Run

In [3]:
queries = [
    "Who developed FIFA 21?",
    "When was God of War Ragnarok released?",
    "What platform was Pokémon Red launched on?",
    "What is Rockstar Games working on right now?",
]

for query in queries:
    print("\n" + "=" * 100)
    print("QUERY:", query)
    run = agent.invoke(query, session_id="udaplay_demo")
    final_state = run.get_final_state()
    report = final_state["final_report"]

    print("\nFINAL TEXT REPORT")
    print(final_state["final_text"])

    print("\nSTRUCTURED JSON REPORT")
    print(format_json_report(report))

    print("\nSNAPSHOTS")
    for snapshot in run.snapshots:
        print("-", snapshot.step_id)


QUERY: Who developed FIFA 21?
[StateMachine] Starting: __entry__
[StateMachine] Executing step: retrieve_game
[StateMachine] Executing step: evaluate_retrieval
[StateMachine] Executing step: final_answer
[StateMachine] Terminating: __termination__

FINAL TEXT REPORT
Answer:
FIFA 21 was developed by EA Vancouver and EA Romania.

Confidence: high
Source Used: local_vector_db
Fallback Used: False

Tools Used:
- retrieve_game
- evaluate_retrieval

Sources:
- FIFA 21 [local]

STRUCTURED JSON REPORT
{
  "question": "Who developed FIFA 21?",
  "answer": "FIFA 21 was developed by EA Vancouver and EA Romania.",
  "confidence": "high",
  "source_used": "local_vector_db",
  "fallback_used": false,
  "tools_used": [
    "retrieve_game",
    "evaluate_retrieval"
  ],
  "retrieval_evaluation": {
    "tool": "evaluate_retrieval",
    "is_sufficient": true,
    "confidence": "high",
    "reason": "The top local result appears relevant and contains the requested information.",
    "top_similarity": 0.

### 3. Check history

In [4]:
runs = agent.get_session_runs("udaplay_demo")
print(f"Total runs in session: {len(runs)}")

for i, run in enumerate(runs, 1):
    report = run.get_final_state()["final_report"]
    print(f"\nRun {i}")
    print("Question:", report["question"])
    print("Confidence:", report["confidence"])
    print("Fallback used:", report["fallback_used"])
    print("Tools used:", " -> ".join(report["tools_used"]))
    print("Answer:", report["answer"])

Total runs in session: 4

Run 1
Question: Who developed FIFA 21?
Confidence: high
Fallback used: False
Tools used: retrieve_game -> evaluate_retrieval
Answer: FIFA 21 was developed by EA Vancouver and EA Romania.

Run 2
Question: When was God of War Ragnarok released?
Confidence: high
Fallback used: False
Tools used: retrieve_game -> evaluate_retrieval
Answer: God of War Ragnarok was released on November 9, 2022.

Run 3
Question: What platform was Pokémon Red launched on?
Confidence: high
Fallback used: False
Tools used: retrieve_game -> evaluate_retrieval
Answer: Pokémon Red was released on February 27, 1996.

Run 4
Question: What is Rockstar Games working on right now?
Confidence: medium
Fallback used: True
Tools used: retrieve_game -> evaluate_retrieval -> game_web_search -> persist_web_memory
Answer: Rockstar Games is currently working on Grand Theft Auto VI, set for release on May 26, 2026. They are also developing a new Max Payne game with Remedy Entertainment.
